# 01 - Atualizacao de parlamentares

Atualiza deputados e senadores com endpoints de detalhe e regenera `parlamentares/v1` antes das faixas da Camara.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
ACTIVE_CONFIG_PATH = DATA_ROOT / "operations" / "atualizacao" / "active.json"
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # Opcional: branch, tag ou commit. Vazio usa o default remoto.

os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
for name in ["raw", "checkpoints", "logs", "manifests", "processed", "operations/atualizacao"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("DATA_ROOT:", DATA_ROOT)
print("Repositorio:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

In [ ]:
EXPECTED_CYCLE_ID = "20260713"
if not ACTIVE_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Controle ativo ausente: {ACTIVE_CONFIG_PATH}. Execute o caderno 00 primeiro.")
CONFIG = json.loads(ACTIVE_CONFIG_PATH.read_text(encoding="utf-8"))
assert CONFIG["schema_version"] == 1
assert CONFIG["cycle_id"] == EXPECTED_CYCLE_ID, CONFIG["cycle_id"]
assert CONFIG["window"] == {"data_inicio": "2026-05-01", "data_fim": "2026-07-13"}
assert CONFIG["data_inicio"] == CONFIG["window"]["data_inicio"]
assert CONFIG["data_fim"] == CONFIG["window"]["data_fim"]
assert Path(CONFIG["data_root"]) == DATA_ROOT
RUNS = {item["key"]: item for item in CONFIG["collection_runs"]}
print("Ciclo ativo:", CONFIG["cycle_id"], CONFIG["window"])

In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone

TERMINAL_STATUSES = {"completed"}
DEFERRED_COLLECTIONS_PATH = (
    DATA_ROOT
    / "operations"
    / "atualizacao"
    / "ciclos"
    / EXPECTED_CYCLE_ID
    / "deferred_collections.json"
)
DEFERRED_COLLECTION_POLICIES = {
    "senado_ccj_historico": {
        "run_id": "prod-historico-senado-ccj",
        "allowed_statuses": ["completed_with_errors"],
        "allowed_unresolved_partitions": ["2015-05"],
        "analysis_exclusion": "senado/ccj_notas",
    }
}

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

def manifest_for(run):
    return DATA_ROOT / "manifests" / f"{run['run_id']}.json"

def checkpoint_for(run):
    return DATA_ROOT / "checkpoints" / run["source"] / f"{run['dataset']}.json"

def unresolved_partitions(run):
    checkpoint = read_json(checkpoint_for(run)) or {}
    current = (checkpoint.get("runs") or {}).get(run["run_id"], {}) or {}
    failed = set((current.get("failed_partitions") or {}).keys())
    completed = set((current.get("completed_partitions") or {}).keys())
    return sorted(failed - completed)

def _assert_manifest_contract(run):
    manifest = read_json(manifest_for(run))
    assert manifest is not None, f"Manifest final ausente: {manifest_for(run)}"
    assert manifest.get("run_id") == run["run_id"]
    assert manifest.get("mode") == "prod", (run["key"], manifest.get("mode"))
    assert manifest.get("sample") is False, (run["key"], manifest.get("sample"))
    assert manifest.get("data_inicio") == run["data_inicio"], (run["key"], manifest.get("data_inicio"))
    assert manifest.get("data_fim") == run["data_fim"], (run["key"], manifest.get("data_fim"))
    return manifest

def assert_collection_complete(run):
    manifest = _assert_manifest_contract(run)
    assert manifest.get("status") in TERMINAL_STATUSES, (run["key"], manifest.get("status"))
    unresolved = unresolved_partitions(run)
    assert not unresolved, f"Particoes falhas nao resolvidas em {run['key']}: {unresolved[:20]}"
    return manifest

def deferred_collection_for(run):
    payload = read_json(DEFERRED_COLLECTIONS_PATH)
    if not payload:
        return None
    assert payload.get("schema_version") == 1, DEFERRED_COLLECTIONS_PATH
    assert payload.get("cycle_id") == EXPECTED_CYCLE_ID, payload.get("cycle_id")
    for item in payload.get("items", []):
        if item.get("key") == run["key"]:
            policy = DEFERRED_COLLECTION_POLICIES.get(run["key"])
            assert policy is not None, f"Adiamento sem politica: {run['key']}"
            assert item.get("run_id") == run["run_id"] == policy["run_id"], item
            assert item.get("allowed_statuses") == policy["allowed_statuses"], item
            assert (
                item.get("allowed_unresolved_partitions")
                == policy["allowed_unresolved_partitions"]
            ), item
            assert item.get("analysis_exclusion") == policy["analysis_exclusion"], item
            return item
    return None

def collection_acceptance(run):
    try:
        manifest = assert_collection_complete(run)
        return {
            "manifest": manifest,
            "deferred": False,
            "status": manifest.get("status"),
            "unresolved": [],
        }
    except AssertionError as strict_error:
        deferral = deferred_collection_for(run)
        assert deferral is not None, strict_error
        manifest = _assert_manifest_contract(run)
        allowed_statuses = set(deferral.get("allowed_statuses") or [])
        allowed_unresolved = sorted(deferral.get("allowed_unresolved_partitions") or [])
        actual_unresolved = unresolved_partitions(run)
        assert deferral.get("analysis_excluded") is True, deferral
        assert str(deferral.get("reason") or "").strip(), deferral
        assert manifest.get("status") in allowed_statuses, (
            run["key"], manifest.get("status"), sorted(allowed_statuses)
        )
        assert actual_unresolved == allowed_unresolved, {
            "run": run["key"],
            "expected_unresolved": allowed_unresolved,
            "actual_unresolved": actual_unresolved,
        }
        return {
            "manifest": manifest,
            "deferred": True,
            "status": manifest.get("status"),
            "unresolved": actual_unresolved,
            "reason": deferral["reason"],
            "follow_up": deferral.get("follow_up"),
        }

def assert_collection_accepted(run):
    return collection_acceptance(run)["manifest"]

def show_run_state(run, tail_lines=5):
    final = read_json(manifest_for(run))
    autosave_path = DATA_ROOT / "manifests" / f"{run['run_id']}.autosave.json"
    autosave = read_json(autosave_path)
    log_path = DATA_ROOT / "logs" / f"{run['run_id']}.jsonl"
    tail = log_path.read_text(encoding="utf-8").splitlines()[-tail_lines:] if log_path.exists() else []
    print(run["key"], {
        "manifest": str(manifest_for(run)),
        "status": final.get("status") if final else None,
        "autosave_status": autosave.get("status") if autosave else None,
        "unresolved": unresolved_partitions(run),
        "log_tail": tail,
    })

def collector_command(run, *extra):
    return [
        sys.executable, "-u", "-m", run["module"],
        "--mode", "prod",
        "--output-dir", str(DATA_ROOT),
        "--data-inicio", run["data_inicio"],
        "--data-fim", run["data_fim"],
        "--run-id", run["run_id"],
        "--no-sample", "--resume", *extra,
    ]

def run_streamed(command, label):
    print(f"\n=== {label} ===", flush=True)
    print(" ".join(map(str, command)), flush=True)
    completed = subprocess.run(list(map(str, command)), check=False)
    returncode = completed.returncode
    print(f"=== retorno {returncode}: {label} ===", flush=True)
    return returncode

@contextmanager
def dataset_lock(run):
    lock_root = DATA_ROOT / "operations" / "atualizacao" / "locks"
    lock_root.mkdir(parents=True, exist_ok=True)
    lock_path = lock_root / f"{run['source']}__{run['dataset']}.json"
    payload = {
        "cycle_id": CONFIG["cycle_id"],
        "run_id": run["run_id"],
        "source": run["source"],
        "dataset": run["dataset"],
        "started_at": datetime.now(timezone.utc).isoformat(),
    }
    try:
        with lock_path.open("x", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
            handle.write("\n")
    except FileExistsError as exc:
        raise RuntimeError(f"Dataset ja bloqueado por outra sessao: {lock_path}\n{lock_path.read_text()}") from exc
    try:
        yield
    finally:
        if lock_path.exists() and read_json(lock_path) == payload:
            lock_path.unlink()

def run_collector(run, *extra):
    with dataset_lock(run):
        return run_streamed(collector_command(run, *extra), run["key"])

def require_explicit_confirmation(enabled, confirmation):
    if enabled:
        assert confirmation == EXPECTED_CYCLE_ID, "Digite o cycle_id na variavel CONFIRMAR_CICLO."

def assert_parlamentares_ready():
    run_id = CONFIG["processing_run_ids"]["parlamentares"]
    manifest_path = DATA_ROOT / "processed" / "manifests" / f"{run_id}-parlamentares.json"
    periodos_path = DATA_ROOT / "processed" / "parlamentares" / "v1" / "parquet" / "parlamentares_periodos.parquet"
    manifest = read_json(manifest_path)
    assert manifest and manifest.get("run_id") == run_id and manifest.get("dataset_version") == "v1", manifest_path
    assert periodos_path.exists(), periodos_path
    return manifest

In [ ]:
RODAR_COLETA = False
RODAR_PROCESSAMENTO = False
CONFIRMAR_CICLO = ""
require_explicit_confirmation(RODAR_COLETA or RODAR_PROCESSAMENTO, CONFIRMAR_CICLO)

run = RUNS["parlamentares"]
if RODAR_COLETA:
    rc = run_collector(run, "--source", "all")
    assert rc == 0
    assert_collection_complete(run)

if RODAR_PROCESSAMENTO:
    command = [
        sys.executable, "-u", "-m", "processamento.parlamentares",
        "--mode", "prod", "--data-root", str(DATA_ROOT),
        "--run-id", CONFIG["processing_run_ids"]["parlamentares"],
        "--data-inicio", CONFIG["historical_floor"],
        "--data-fim", CONFIG["window"]["data_fim"], "--overwrite",
    ]
    assert run_streamed(command, "processar parlamentares/v1 current") == 0

## Gate de parlamentares e mandatos

In [ ]:
import pyarrow.parquet as pq

show_run_state(RUNS["parlamentares"])
assert_collection_complete(RUNS["parlamentares"])
proc_run = CONFIG["processing_run_ids"]["parlamentares"]
proc_manifest_path = DATA_ROOT / "processed" / "manifests" / f"{proc_run}-parlamentares.json"
proc_manifest = read_json(proc_manifest_path)
assert proc_manifest and proc_manifest.get("dataset_version") == "v1"
periodos_path = DATA_ROOT / "processed" / "parlamentares" / "v1" / "parquet" / "parlamentares_periodos.parquet"
assert periodos_path.exists()
table = pq.read_table(periodos_path, columns=["parlamentar_id", "source", "vigencia_inicio", "vigencia_fim"])
assert table.num_rows > 0
rows = table.to_pylist()
assert all(row["parlamentar_id"] and row["source"] and row["vigencia_inicio"] for row in rows)
invalid = [row for row in rows if row.get("vigencia_fim") and row["vigencia_inicio"] > row["vigencia_fim"]]
assert not invalid, invalid[:10]
print("Gate aprovado:", table.num_rows, "periodos de mandato")